# Analisis de Resultados de YOHO

## Objetivo

Este notebook interpreta los resultados del entrenamiento de YOLO11n sobre el dataset de espectrogramas de eventos sonoros. Se evalua el modelo sobre el conjunto de test (reservado durante el entrenamiento), se analizan las metricas globales y por clase, y se identifican los patrones de exito y fracaso para la discusion del informe.

## Contexto del entrenamiento

- Modelo: YOLO11n
- Dataset: 2,848 imagenes de train, 346 de val, 354 de test (aproximadamente)
- Entrada: espectrogramas de 1 canal (escala de grises, 16 bits)
- 22 clases de eventos sonoros
- 100 epocas, 3.17 horas de entrenamiento
- mAP@0.5 en validacion: 0.694

In [ ]:
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from collections import Counter

sys.path.append("../src")
from config import OUTPUT_ROOT, SPECTROGRAM_DIR, LABELS_DIR
# Estilo oscuro global
plt.style.use('dark_background')

## 1. Cargar el modelo entrenado y evaluar sobre test

El conjunto de test no se uso durante el entrenamiento ni la validacion. Estas metricas son la evaluacion final e imparcial del modelo.

In [ ]:
from ultralytics import YOLO

best_pt = OUTPUT_ROOT / "runs" / "yoho_train" / "weights" / "best.pt"
print(f"Modelo: {best_pt}")
print(f"Existe: {best_pt.exists()}")

model = YOLO(str(best_pt))

metrics_test = model.val(
    data=str(OUTPUT_ROOT / "data.yaml"),
    split="test",
    imgsz=1024,
    batch=8,
    device=0,
    project=str(OUTPUT_ROOT / "runs"),
    name="yoho_test",
    exist_ok=True,
)

print(f"\nMetricas sobre TEST:")
print(f"  mAP@0.5:      {metrics_test.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {metrics_test.box.map:.4f}")
print(f"  Precision:    {metrics_test.box.mp:.4f}")
print(f"  Recall:       {metrics_test.box.mr:.4f}")

## 2. Metricas por clase

Se extrae el mAP@0.5 por clase para identificar las clases con mejor y peor rendimiento. Esto permite analizar que tipos de eventos son faciles o dificiles para el modelo.

In [ ]:
# Extraer nombres de clases desde el data.yaml
with open(OUTPUT_ROOT / "data.yaml", 'r') as f:
    data_yaml = yaml.safe_load(f)
names = data_yaml['names']
if isinstance(names, dict):
    clase_id_a_nombre = {i: names[i] for i in sorted(names.keys())}
else:
    clase_id_a_nombre = {i: n for i, n in enumerate(names)}

# Construir DataFrame de metricas por clase
clases_ids = []
map50_vals = []
map_vals = []
precision_vals = []
recall_vals = []

for i, clase_id in enumerate(metrics_test.box.ap_class_index):
    clases_ids.append(int(clase_id))
    map50_vals.append(float(metrics_test.box.ap50[i]))
    map_vals.append(float(metrics_test.box.ap[i]))

df_metricas = pd.DataFrame({
    'clase_id': clases_ids,
    'clase': [clase_id_a_nombre[c] for c in clases_ids],
    'mAP50': map50_vals,
    'mAP50-95': map_vals,
})
df_metricas = df_metricas.sort_values('mAP50', ascending=False).reset_index(drop=True)

print("Metricas por clase (ordenadas por mAP50):")
print(df_metricas.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
colores = ['#4ECDC4' if v >= 0.7 else '#FFE66D' if v >= 0.5 else '#FF6B6B'
           for v in df_metricas['mAP50']]
ax.barh(range(len(df_metricas)), df_metricas['mAP50'], color=colores)
ax.set_yticks(range(len(df_metricas)))
ax.set_yticklabels(df_metricas['clase'], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("mAP@0.5")
ax.set_title("mAP@0.5 por clase (test)")
ax.axvline(metrics_test.box.map50, color='white', linestyle='--',
           label=f"Global: {metrics_test.box.map50:.3f}")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 3. Curvas de entrenamiento

Se revisan las curvas de loss y metricas por epoca para verificar si el modelo convergio o si hubo sobreajuste. Si la loss de validacion empieza a subir mientras la de train sigue bajando, hay sobreajuste.

In [ ]:
results_img = OUTPUT_ROOT / "runs" / "yoho_train" / "results.png"
if results_img.exists():
    img = Image.open(results_img)
    plt.figure(figsize=(14, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Curvas de entrenamiento (results.png)")
    plt.tight_layout()
    plt.show()
else:
    print(f"No encontrado: {results_img}")

## 4. Matriz de confusion

La matriz de confusion muestra cuantas predicciones de cada clase son correctas y cuales se confunden con otras clases. Es clave para entender los errores sistematicos, especialmente entre clases acusticamente similares.

In [ ]:
for nombre in ['confusion_matrix.png', 'confusion_matrix_normalized.png']:
    cm_path = OUTPUT_ROOT / "runs" / "yoho_train" / nombre
    if cm_path.exists():
        img = Image.open(cm_path)
        plt.figure(figsize=(14, 12))
        plt.imshow(img)
        plt.axis('off')
        plt.title(nombre)
        plt.tight_layout()
        plt.show()
    else:
        print(f"No encontrado: {cm_path}")

## 5. Curvas Precision-Recall, Precision, Recall y F1

Estas curvas muestran el trade-off entre precision y recall para cada clase. Las clases con curvas PR planas (area pequeña) son las que el modelo no logra separar bien.

In [ ]:
for nombre in ['PR_curve.png', 'P_curve.png', 'R_curve.png', 'F1_curve.png']:
    p = OUTPUT_ROOT / "runs" / "yoho_train" / nombre
    if p.exists():
        img = Image.open(p)
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(nombre)
        plt.tight_layout()
        plt.show()

## 6. Predicciones sobre imagenes de test

Se visualizan las predicciones del modelo sobre algunas imagenes del conjunto de test. Se comparan las cajas predichas (colores) con las cajas reales (ground truth) para identificar aciertos y errores visualmente.

In [ ]:
def visualizar_prediccion(img_path, conf_threshold=0.25):
    """
    Ejecuta el modelo sobre una imagen y muestra:
    - Izquierda: imagen con cajas reales (ground truth).
    - Derecha: imagen con cajas predichas.
    """
    img_path = Path(img_path)
    label_path = LABELS_DIR / "test" / (img_path.stem + ".txt")

    # Leer ground truth
    gt_lineas = []
    if label_path.exists():
        with open(label_path, 'r') as f:
            gt_lineas = [l.strip() for l in f if l.strip()]

    # Prediccion
    resultados = model.predict(
        source=str(img_path),
        imgsz=1024,
        conf=conf_threshold,
        device=0,
        verbose=False,
    )
    r = resultados[0]
    cajas_pred = r.boxes

    img = Image.open(img_path)
    ancho, alto = img.size

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    # Panel izquierdo: ground truth
    ax = axes[0]
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Ground truth ({len(gt_lineas)} cajas)")
    for linea in gt_lineas:
        partes = linea.split()
        cid = int(partes[0])
        x_c, y_c, w, h = map(float, partes[1:])
        x0 = (x_c - w/2) * ancho
        y0 = (y_c - h/2) * alto
        rect = patches.Rectangle((x0, y0), w*ancho, h*alto,
                                 linewidth=2, edgecolor='#4ECDC4',
                                 facecolor='#4ECDC4', alpha=0.2)
        ax.add_patch(rect)
        ax.text(x0 + 5, y0 + 15, clase_id_a_nombre.get(cid, str(cid)),
                fontsize=7, color='white',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#4ECDC4',
                          alpha=0.8, edgecolor='none'))
    ax.axis('off')

    # Panel derecho: predicciones
    ax = axes[1]
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Predicciones ({len(cajas_pred)} cajas, conf>={conf_threshold})")
    for box in cajas_pred:
        xyxy = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        cid = int(box.cls[0])
        x0, y0, x1, y1 = xyxy
        rect = patches.Rectangle((x0, y0), x1-x0, y1-y0,
                                 linewidth=2, edgecolor='#FF6B6B',
                                 facecolor='#FF6B6B', alpha=0.2)
        ax.add_patch(rect)
        nombre = clase_id_a_nombre.get(cid, str(cid))
        ax.text(x0 + 5, y0 + 15, f"{nombre} {conf:.2f}",
                fontsize=7, color='white',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#FF6B6B',
                          alpha=0.8, edgecolor='none'))
    ax.axis('off')

    fig.suptitle(img_path.name, fontsize=12)
    plt.tight_layout()
    plt.show()

    return r

In [ ]:
test_imgs = sorted((SPECTROGRAM_DIR / "test").glob("*.png"))
print(f"Imagenes de test: {len(test_imgs)}")

# Elegir 4 imagenes al azar (semilla fija para reproducibilidad)
import random
random.seed(1932)
seleccion = random.sample(test_imgs, min(4, len(test_imgs)))

for img_path in seleccion:
    visualizar_prediccion(img_path)

In [ ]:
test_imgs = sorted((SPECTROGRAM_DIR / "test").glob("*.png"))
print(f"Imagenes de test: {len(test_imgs)}")

# Elegir 4 imagenes al azar (semilla fija para reproducibilidad)
import random
random.seed(135612)
seleccion = random.sample(test_imgs, min(4, len(test_imgs)))

for img_path in seleccion:
    visualizar_prediccion(img_path)

## 7. Analisis de errores

A partir de las metricas por clase y la matriz de confusion, se identifican patrones:

- Clases con mAP alto: eventos sostenidos con firma espectral clara.
- Clases con mAP bajo: eventos breves, transitorios, o acusticamente similares a otras clases.

Se documentan las confusiones mas frecuentes y se proponen hipotesis para explicarlas.

In [ ]:
top5 = df_metricas.head(5)
bottom5 = df_metricas.tail(5)

print("TOP 5 clases con mejor mAP@0.5:")
print(top5[['clase', 'mAP50', 'mAP50-95']].to_string(index=False))

print("\nTOP 5 clases con peor mAP@0.5:")
print(bottom5[['clase', 'mAP50', 'mAP50-95']].to_string(index=False))